# Fine-tune smoke test (pipeline check)

Purpose
-------

- Verify processed WAVs + metadata are ready for training end-to-end;
- Create mel spectograms for a small subset, check shapes and dtypes;
- Optionally call a trainer script for 1-2 steps (if you have one) to confirm training starts.

Run this after the metadata.csv is filled and you ran the validation.

## Imports and consts

In [ ]:
from pathlib import Path
import soundfile as sf
import numpy as np
from TTS.api import TTS

try:
    import librosa
except ImportError:
    librosa = None

PROCESSED_DIR = Path("data/processed/wavs")
METADATA = Path("data/processed/metadata.csv")
SAMPLES_DIR = Path("data/outputs/smoke_samples")
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SAMPLE_RATE = 22050
NUMBER_OF_TESTS = 8

MODEL_NAME = "tts_models/pt/cv/vits"
OUTPUT_DIR = Path("data/outputs/smoke_tts")

## Selecting Test Entries from Metadata

This cell below loads the metadata file and select the first `NUMBER_OF_TESTS` entries that have non-empty transcriptions. 
We will process only this small set to create mel spectograms and verify shapes.

In [ ]:
metadata = []

if not METADATA.exists():
    raise FileNotFoundError(f"{METADATA} not found. Run preprocessing first.")

with open(METADATA, "r", encoding="utf-8") as file:
    for line_raw in file:
        parts = [part.strip() for part in line_raw.rstrip("\n").split("|")]
        
        if not parts:
            continue
        if len(parts) == 1:
            parts = [parts[0], "", ""]
        elif len(parts) == 2:
            parts.append("")
        metadata.append((parts[0], parts[1], parts[2]))
        
selected = [row for row in metadata if row[1].strip()]
selected = selected[:NUMBER_OF_TESTS]
print(f"Selected {len(selected)} sample(s) for smoke test.")
for fn, text, spk in selected:
    print(" ", fn, "|", text[:80])            

## Creting the mel spectograms fro each WAV:

- Resample if needed, convert to mono, normalize RMS (best effort);
- Compute mel using librosa (if installed) or a fallback STFT -> mel matrix;
- Save a small numpy file per example so you can inspect shapes/dtypes.

In [ ]:
def load_mono_sample(path, sample_rate=TARGET_SAMPLE_RATE):
    data, sample_rate_file = sf.read(str(path), always_2d=True)
    wav = np.mean(data, axis=1)
    if sample_rate_file != sample_rate:
        if librosa is not None:
            wav = librosa.resample(wav, orig_sr=sample_rate_file, target_sr=sample_rate)
        else:
            old_idx = np.linspace(0, len(wav)-1, num=len(wav))
            new_len = int(round(len(wav) * sample_rate / sample_rate_file))
            new_idx = np.linspace(0, len(wav)-1, num=max(1,new_len))
            wav = np.interp(new_idx, old_idx, wav)
            
    return wav, sample_rate

def wav_to_mel(wav, sample_rate = TARGET_SAMPLE_RATE, n_mels = 80, hop_length = 256, win_length = 1024):
    if librosa is not None:
        mel = librosa.feature.melspectrogram(y=wav, sr=sample_rate, n_mels=n_mels, n_fft=win_length, hop_length=hop_length, power=1.0)
        mel = np.log(np.maximum(1e-5, mel))
        return mel.astype(np.float32)

    import scipy.signal
    frequencies, times, stft_matrix = scipy.signal.stft(wav, fs=sample_rate, nperseg=win_length, noverlap=win_length - hop_length)
    magnitude = np.abs(stft_matrix)
    mel = np.abs(np.linspace(0, magnitude.shape[0]-1, n_mels, dtype=int))
    mel_spec = magnitude[mel, :]
    mel_spec = np.log(np.maximum(1e-5, mel_spec))
    return mel_spec.astype(np.float32)

saved = []
for filename, transcript, speaker in selected:
    wav_path = PROCESSED_DIR / filename
    if not wav_path.exists():
        print("MISSING:", wav_path)
        continue

    waveform, sr = load_mono_sample(wav_path, TARGET_SAMPLE_RATE)
    mel_spec = wav_to_mel(waveform, sr)

    out_path = SAMPLES_DIR / (wav_path.stem + ".npy")
    np.save(str(out_path), mel_spec)

    saved.append((filename, out_path, mel_spec.shape))
    print("Saved mel:", filename, "->", out_path.name, "shape", mel_spec.shape)
    
print("Saved", len(saved), "mel files to", SAMPLES_DIR)


## Inspect the saved mel files:

- Open a few `.npy` files in Python or in the notebook to verify shapes and that values are finite;
- Confirm mel shapes are roughly consistent across similarly long utterances.

In [ ]:
for filename, out_path, mel_shape in saved:
    mel = np.load(str(out_path))
    print(out_path.name, "shape:", mel.shape, "min/max:", mel.min(), mel.max(), "mean:", float(mel.mean()))

## Synthesis with VITS (tts_models/pt/cv/vits)

This cell synthesizes the transcripts selected earlier using the prebuilt VITS model `tts_models/pt/cv/vits` (Coqui TTS API).

What it does:

- Loads the TTS model (will download if not present);
- For each selected metadata entry (the `selected` list), synthesize the transcription and write a WAV to `data/outputs/smoke_tts/`;
- Save messages and basic diagnostics.

Notes:

- This is inference only - not fine-tuning.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    print("Loading model:", MODEL_NAME)
    tts = TTS(model_name=MODEL_NAME)
    sample_rate = getattr(getattr(tts, "synthesizer", None), "output_sample_rate", 22050)
except Exception as error:
    raise RuntimeError(f"Failed to load TTS model {MODEL_NAME}: {error}") from error

saved_audio = []

for file_name, text, speaker in selected:
    if not text or not text.strip():
        print("SKIP (no transcription):", file_name)
        continue
    try:
        wav = tts.tts(text)
        output_path = OUTPUT_DIR / f"{Path(file_name).stem}_vits.wav"
        sf.write(str(output_path), wav, sample_rate)
        saved_audio.append(output_path)
        print("Wrote:", output_path.name, "sample_rate:", sample_rate)
    except Exception as error:
        print("SYNTH FAIL:", file_name, "->", error)

print(f"Synthesis complete. {len(saved_audio)} files written to {OUTPUT_DIR}")
